# EM synapse mapping - multi-neuron use case

Creating a circuit from a set of morphologies with spines.

# Authentication

Follow the instructions below to authenticate and select a project to work in.

The morphology-with-spines instances that are to serve as the input of circuit generation must be accessible from the project.

In [ ]:
import obi_auth

from obi_notebook.get_projects import get_projects
from obi_notebook.get_entities import get_entities

from entitysdk.models import CellMorphology, EMCellMesh, EMDenseReconstructionDataset, Circuit
from entitysdk import Client
from entitysdk._server_schemas import AssetLabel

token = obi_auth.get_token(environment="production", auth_mode="daf")
project_context = get_projects(token, env="production")

# Find skeletonized morphologies as input

We search for morphologies originating from the `IARPA MICrONS mouse`, as they are derived by skeletonization

In [ ]:
entity_client = Client(token_manager=token, project_context=project_context, environment="production")

results = entity_client.search_entity(
    entity_type=CellMorphology,
    query={"subject__name": "IARPA MICrONS mouse"},
).all()

# Filter by morphology name, avoid duplicates
morph_names = []
spiny_morphologies = []
for morph in results:
    if morph.name not in morph_names:
        morph_names.append(morph.name)
        spiny_morphologies.append(morph)

print(f"Found {len(spiny_morphologies)} spiny morphologies:")
for m in spiny_morphologies:
    print(f"  - {m.name} (id={m.id})")


## Get `pt_root_id`. 
Next, we have to link the skeletons to their originating cell mesh. In the absence of an activity linking the two, we match them via the identifier, `pt_root_id`.

We print the description of the neurons below.

In [ ]:
from entitysdk.models import SkeletonizationExecution, EMCellMesh
from obi_one.scientific.from_id.cell_morphology_from_id import CellMorphologyFromID
from obi_one.scientific.tasks.em_synapse_mapping.resolve_neuron import resolve_provenance

# {pt_root_id: {"morph_from_id": CellMorphologyFromID, "name": str}}
neurons = {}

for morph_entity in spiny_morphologies:
    morph_from_id = CellMorphologyFromID(id_str=str(morph_entity.id))

    if not morph_from_id.has_source_mesh(db_client=entity_client):
        print(f"  WARNING: Could not resolve provenance for {morph_entity.name}, skipping.")
        continue

    pt_root_id, _, _ = resolve_provenance(entity_client, morph_from_id)

    print(f"{morph_entity.description}")
    print(f"  {morph_entity.name} -> pt_root_id={pt_root_id}")
    neurons[pt_root_id] = {"morph_from_id": morph_from_id, "name": morph_entity.name}

print(f"\nTotal neuron entries: {len(neurons)}")

## Inspect connectivity between neurons

Query CAVE for synapses among the resolved neurons and show, for each physical neuron, which other neurons it connects to (pre- and post-synaptically).

First, create a CAVE token

In [ ]:
import caveclient
temporary_client = caveclient.CAVEclient()
temporary_client.auth.get_new_token()

Paste your token below:

In [ ]:
import os
os.environ["CAVE_TOKEN"] = "PASTE_YOUR_TOKEN_HERE"

Inspect connectivity between the neurons in the set.

In [ ]:
from collections import defaultdict

# Ignore caveclient/materializationengine.py:1745 -- RuntimeWarning
import warnings
warnings.filterwarnings("ignore", message="Engine has switched to", category=RuntimeWarning)

CAVE_DATASTACK = "minnie65_public"

cave = caveclient.CAVEclient(CAVE_DATASTACK, auth_token=os.environ["CAVE_TOKEN"])

pt_root_ids = list(neurons.keys())
pt_root_set = set(pt_root_ids)

# For each neuron, query its afferent synapses and check which are from our set
connectivity = defaultdict(lambda: defaultdict(int))  # {post_id: {pre_id: count}}

for pt_id in pt_root_ids:
    syns = cave.materialize.synapse_query(post_ids=[pt_id])
    internal = syns[syns["pre_pt_root_id"].isin(pt_root_set)]
    for pre_id, count in internal["pre_pt_root_id"].value_counts().items():
        if pre_id != pt_id:  # exclude autapses
            connectivity[pt_id][pre_id] = count

# Print connectivity summary
print("Connectivity between neurons:")
print("=" * 60)
no_int_conns = []
for pt_id in pt_root_ids:
    name = neurons[pt_id]["name"]
    partners = connectivity[pt_id]
    if partners:
        partner_strs = [
            f"{neurons[pid]['name']} ({n} synapses)"
            for pid, n in sorted(partners.items(), key=lambda x: -x[1])
        ]
        print(f"\n{name} (pt_root_id={pt_id}) receives from:")
        for s in partner_strs:
            print(f"  <- {s}")
    else:
        no_int_conns.append((name, pt_id))

if no_int_conns:
    print("\n\nThe following neurons don't have internal incoming connections")
    print("=" * 60)
    for name, pt_id in no_int_conns:
        print(f"  {name} (pt_root_id={pt_id})")

Now we pick the pair of neurons with the highest connectivity between them.

In [ ]:
best_pair = None
best_count = 0

for post_id, partners in connectivity.items():
    for pre_id, count in partners.items():
        # Sum pre and post
        total = count + connectivity[pre_id].get(post_id, 0)
        if total > best_count:
            best_count = total
            best_pair = (pre_id, post_id)

if best_pair:
    a, b = sorted(best_pair)
    print(f"Best connected pair: {neurons[a]['name']} <-> {neurons[b]['name']}")
    print(f"  {neurons[a]['name']} -> {neurons[b]['name']}: {connectivity[b].get(a, 0)} synapses")
    print(f"  {neurons[b]['name']} -> {neurons[a]['name']}: {connectivity[a].get(b, 0)} synapses")
    print(f"  Total: {best_count}")

    # Select the corresponding neuron entries
    circuit_pt_root_ids = list(best_pair)
    circuit_neurons = [neurons[pt_id]["morph_from_id"] for pt_id in circuit_pt_root_ids]
else:
    print("No internal connections found between any neurons.")

In [ ]:
circuit_pt_root_ids.append(post_id)
circuit_neurons = [neurons[pt_id]["morph_from_id"] for pt_id in circuit_pt_root_ids]
print(circuit_pt_root_ids)

# Set up the multi-neuron synapse mapping task

In [ ]:
from obi_one.scientific.tasks.em_synapse_mapping.config import EMSynapseMappingSingleConfig
from obi_one.scientific.tasks.em_synapse_mapping.task import EMSynapseMappingTask
from obi_one.core.info import Info

# We take the last 3 digits of pt_root_id to easily identify the neurons
circuit_neurons_short_names = [str(pt_id)[-3:] for pt_id in circuit_pt_root_ids]

OUTPUT_DIR = "multi-neuron-circuit-" + "-".join([id for id in circuit_neurons_short_names])

task_config = EMSynapseMappingSingleConfig(
    info=Info(
        campaign_name="EM Synapse Mapping Multi-Neuron",
        campaign_description="Map EM synapses onto multiple spiny morphologies"
    ),
    initialize=EMSynapseMappingSingleConfig.Initialize(
        neurons=circuit_neurons,
    ),
    coordinate_output_root=OUTPUT_DIR,
)

task = EMSynapseMappingTask(config=task_config)
print(f"Running synapse mapping task with {len(circuit_neurons)} neurons...")
print(f"Output: {OUTPUT_DIR}/")
task.execute(db_client=entity_client)

## Inspect the output (local files)

In [ ]:
import json
import h5py
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
config_path = output_dir / "circuit_config.json"

if config_path.exists():
    with open(config_path) as f:
        circuit_cfg = json.load(f)
    print(json.dumps(circuit_cfg, indent=2))

    edges_file = output_dir / "synaptome-edges.h5"
    if edges_file.exists():
        with h5py.File(edges_file, "r") as h5:
            print("\nEdge populations:")
            for pop_name in h5["edges"]:
                grp = h5[f"edges/{pop_name}"]
                n = len(grp["source_node_id"])
                src_pop = grp["source_node_id"].attrs.get("node_population", "?")
                tgt_pop = grp["target_node_id"].attrs.get("node_population", "?")
                print(f"  {pop_name}: {n} edges ({src_pop} -> {tgt_pop})")
else:
    print("No circuit output found. Run the task cell above first.")